# Where does the money actually get spent? (contract cross-check via makegov)

The [White House account notebook](white_house_account.ipynb) showed, from OMB's budget data, that the "Repair and Restoration" account (EOP `011-X-0109`) ballooned to \$323M — almost all reimbursable / collected funds. But OMB's apportionment and execution data stops at the *account*. It doesn't tell you what got contracted, or to whom.

To see that, you have to cross to the **contract** record — FPDS — which this notebook queries through [makegov](https://www.makegov.com/)'s Tango API, and check the linkage in [USAspending](https://www.usaspending.gov/).

> **Note on access.** Unlike the free, CC0 BlazingStar data used elsewhere in this repo, **makegov/Tango is a commercial API** — you need an account and an API key. Set it as `TANGO_API_KEY`. (USAspending, used at the end, is free and open.) This notebook won't run without the key, and it's intentionally left out of the repo's CI for that reason.

In [ ]:
%pip install -q tango-python requests pandas

In [ ]:
import os

import pandas as pd
import requests
from tango import TangoClient

api_key = os.environ.get("TANGO_API_KEY")
assert api_key, "Set TANGO_API_KEY — a makegov API key (https://www.makegov.com/). This notebook needs it to query FPDS."
client = TangoClient(api_key=api_key)

## 1. White House renovation contracts (FPDS, via makegov)

The repair-and-restoration work is contracted out. Search FPDS for EOP-funded contracts with "renovation" in the description.

In [ ]:
resp = client.list_contracts(
    funding_agency="Executive Office of the President",
    keyword="renovation",
    order="desc", sort="award_date", limit=20,
)
rows = [{
    "award_date": str(r.get("award_date"))[:10],
    "piid": r.get("piid"),
    "value": float(r.get("total_contract_value") or 0),
    "recipient": (r.get("recipient") or {}).get("display_name"),
    "description": (r.get("description") or "")[:55],
} for r in resp.results]
print("matching contracts:", resp.count)
pd.DataFrame(rows)

## 2. Who awards them, and who funds them

The PIIDs start with **47P** — that prefix is **GSA** (the Public Buildings Service does the contracting for the White House complex). So these are *awarded by GSA* and *funded by EOP*. Confirm one in USAspending — the $635K bathroom renovation at 712 Jackson Place (part of the White House complex).

In [ ]:
PIID = "47PH5426F0109"   # BATHROOM RENOVATION AT 712 JACKSON PLACE
S = requests.Session()
S.headers.update({"User-Agent": "blazingstar-data-demo"})

r = S.post("https://api.usaspending.gov/api/v2/search/spending_by_award/",
           json={"filters": {"keywords": [PIID], "award_type_codes": ["A", "B", "C", "D"]},
                 "fields": ["Award ID", "Recipient Name", "Awarding Agency", "Funding Agency",
                            "generated_internal_id"], "limit": 5}, timeout=60)
award = r.json()["results"][0]
print("Award:    ", award["Award ID"])
print("Recipient:", award["Recipient Name"])
print("Awarded by:", award["Awarding Agency"], " | Funded by:", award["Funding Agency"])

# Account-level (File C) linkage — which Treasury account actually paid?
gid = award["generated_internal_id"]
f = S.post("https://api.usaspending.gov/api/v2/awards/funding/",
           json={"award_id": gid, "limit": 10, "page": 1,
                 "sort": "reporting_fiscal_date", "order": "desc"}, timeout=60)
funding = f.json().get("results", [])
print("\nTreasury-account funding records linked to this award:", len(funding))
for rec in funding:
    print("  ", rec.get("federal_account"), rec.get("treasury_account_symbol"), rec.get("funding_agency_name"))

## What this shows

- **The contracts are public.** GSA awards White House-complex renovation work — e.g. a \$635K bathroom renovation at 712 Jackson Place — and USAspending records EOP as the funder. So the money does *not* vanish.
- **But you can't tie it to the specific account.** The award's account-level (File C) linkage in USAspending is empty, and FPDS carries no Treasury account at all. So the \$323M "Repair and Restoration" apportionment and these visible contracts **can't be rigorously connected in public data** — you can see the budget swell and you can see GSA awarding renovation work, but not that *this* contract was paid from *that* account.
- **The contracting runs through GSA**, which fits the reimbursable / Economy-Act funding seen on the budget side: EOP's account takes money in, and GSA does the buying.
- For reference, a keyword search for `ballroom` returns **no** federal contracts by that description — the marquee project doesn't appear in FPDS under that name.

**Bottom line:** the budget side (BlazingStar/OMB) and the contract side (FPDS/USAspending) are both visible, but the join between them — account → award — is missing for this account. Two halves of the story you can't yet connect with public data alone.